# Requests

`Starlette` 包含一个请求（Request）类，它为你提供了一个更好的接口来接收请求，而不是直接访问 ASGI 范围和接收通道。

## Request
签名 : `Request(scope, receive=None)`


In [ ]:
from starlette.requests import Request
from starlette.responses import Response


async def app(scope, receive, send):
    assert scope['type'] == 'http'
    request = Request(scope, receive)
    content = '%s %s' % (request.method, request.url.path)
    response = Response(content, media_type='text/plain')
    await response(scope, receive, send)

## `scope`、`receive` 和 `send` 是三个核心参数


1. scope (字典)
- 作用：包含当前请求的元信息（类似于 WSGI 中的 environ），但结构更清晰。
- 内容：
    - 固定字段（所有请求类型共有）：
        - `type`：请求类型（如 `'http'`、`'websocket'`）。
        - `asgi`：ASGI 版本（如 `{'version': '3.0'}`）。
    - HTTP 请求特有字段（当 type='http' 时）：
        - `method`：HTTP 方法（如 'GET'、'POST'）。
        - `path`：请求路径（如 '/foo'）。
        - `headers`：请求头（列表形式的二进制键值对，如 [(b'host', b'example.com')]）。
        - 其他（如 scheme、query_string、client 等）。
- 示例：

In [ ]:

assert scope['type'] == 'http'  # 确保是 HTTP 请求
print(scope['method'], scope['path'])  # 如 'GET', '/hello'

2. receive (异步函数)
- 作用：用于从客户端接收消息的异步函数（如 HTTP 请求的请求体）。
- 行为：
    - 调用 await receive() 会返回一个字典形式的事件（event）。
    - 对于 HTTP 请求，常见事件：
       -  `{'type': 'http.request', 'body': b'...', 'more_body': True/False}`
        （more_body 表示是否还有后续数据流）。
- 示例：

In [ ]:
body = b''
while True:
    event = await receive()
    body += event.get('body', b'')
    if not event.get('more_body', False):
        break

3. send (异步函数)
- 作用：向客户端发送消息的异步函数（如 HTTP 响应的状态码、头部和响应体）。
- 行为：
    - 通过 await send(event) 发送事件（event）到客户端。
    - 对于 HTTP 响应，需按顺序发送以下事件：
      - 响应头事件（必须首先发送）：
         ```python
            await send({
                'type': 'http.response.start',
                'status': 200,
                'headers': [(b'content-type', b'text/plain')],
            })
         ```
      - 响应体事件（可多次发送，分块传输）：
        ```python
            await send({
                'type': 'http.response.body',
                'body': b'Hello World',
                'more_body': False  # 是否还有后续数据
            })
         ```

请求提供了一个映射接口，因此您可以像使用 `scope`

例如`request['path']` 将返回 ASGI 路径。

如果不需要访问请求体，则可以实例化一个请求，而不提供`receive`参数。

## 请求解析
在Endpoint中的`request`参数是Starlette中的Request类的实例，其中包含了对HTTP请求的全部解析内容。对于请求的内容可以通过以下方式访问。

- `request.method`，访问方法。
- `request.url`，请求URL，可以通过path、port等属性获取URL的不同部分。
- `request.headers`，字典类型，获取请求头中的内容。
- `request.query_params`，字典类型，获取Query String中的查询参数。
- `request.path_params`，字典类型，获取URL路径中的参数。
- `request.client`，获取客户端的信息，例如`request.client.host`。
- `request.cookies`，获取Cookie的值，使用`.get(name)`方法访问, 例如：`request.cookies.get('mycookie')`。
- `request.body()`，以字节方式异步获取请求体内容。
    - `await request.body()` 以字节为单位的请求正文
    - `async with request.form() as form` 请求正文，解析为表单数据或多部分：
    - `await request.json()` 以 JSON 格式解析的请求正文
    - 您还可以使用 async for 语法，以流的形式访问请求正文：
       ```python
        from starlette.requests import Request
        from starlette.responses import Response
        async def app(scope, receive, send):
            assert scope['type'] == 'http'
            request = Request(scope, receive)
            body = b''
            async for chunk in request.stream():
                body += chunk
            response = Response(body, media_type='text/plain')
            await response(scope, receive, send)
       ```
       - 您还可以使用 async for 语法，以流的形式访问请求正文：
        ```python
        from starlette.requests import Request
        from starlette.responses import Response


        async def app(scope, receive, send):
            assert scope['type'] == 'http'
            request = Request(scope, receive)
            body = b''
            async for chunk in request.stream():
                body += chunk
            response = Response(body, media_type='text/plain')
            await response(scope, receive, send)
        ```
        如果访问 `.stream()`，则会提供字节块，而不会将整个正文存储到内存中。随后对 `.body()`、`.form()` 或 `.json() `的任何调用都会引发错误。
- `request.form()`，异步方式解析表单或者Multipart表单数据。
- `request.json()`，异步方式以JSON格式解析请求体。


### Request Files
请求文件通常作为多部分表单数据（multipart/form-data）发送。      
签名； `request.form(max_files=1000, max_fields=1000, max_part_size=1024*1024)`

您可以使用参数 `max_files` 和 `max_fields` 配置最大字段或文件数量，使用 `max_part_size` 配置部件大小：

```python
async with request.form(max_files=1000, max_fields=1000, max_part_size=1024*1024):
    ...
```
> 这些限制是出于安全考虑，如果允许无限数量的字段或文件，在解析过多的空字段时会消耗大量 CPU 和内存，从而导致拒绝服务攻击。

当您使用 `request.form()` 作为表单调用 `async` 时，您会收到一个 `starlette.datastructures.FormData`，它是一个不可变的多数据文件，包含文件上传和文本输入。文件上传项目表示为 `starlette.datastructures.UploadFile` 的实例。

`UploadFile` 具有以下属性：
- `filename`：包含上传的原始文件名的字符串，如果没有，则为 None（例如 myimage.jpg）。
- `content_type`：一个包含内容类型（MIME 类型/媒体类型）的字符串，如果没有（如 image/jpeg），则为 "无"。
- `file` ：一个 `SpooledTemporaryFile`（类文件对象）。这是一个实际的 Python 文件，您可以直接将它传递给其他函数或库，这些函数或库需要一个 "file-like"对象。
- `headers`：`Headers`对象。通常这只是 `Content-Type` 标头，但如果多部分字段中包含其他标头，它们也会包含在这里。请注意，这些标头与 `Request.headers` 中的标头没有任何关系。
- `size`：以字节为单位表示上传文件大小的 `int`。该值根据请求内容计算得出，因此与 Content-Length 头信息相比，它是查找上传文件大小的最佳选择。如果未设置，则为 "无"。


`UploadFile` 具有以下异步方法。它们都调用下面的相应文件方法（使用内部 SpooledTemporaryFile）。
- `async write（data）`：将数据 （字节） 写入文件。
- `async read（size`）：读取文件的大小 （int） 字节。
- `async seek（offset）`：转到文件中的字节位置 offset （int）。
    - 例如，await myfile.seek（0） 将转到文件的开头。
- `async close（）`：关闭文件。

由于所有这些方法都是异步方法，因此需要 "`await` "它们。 例如，您可以用以下方法获取文件名和内容：

```python
async with request.form() as form:
filename = form["upload_file"].filename
contents = await form["upload_file"].read()
```

如 RFC-7578: 4.2 所述，包含文件的表单数据内容部分假定在 Content-Disposition 标头中有名称和文件名字段：Content-Disposition: form-data; name="user"; filename="somefile".虽然根据 RFC-7578 的规定，文件名字段是可选的，但它有助于 Starlette 区分哪些数据应被视为文件。如果提供了文件名字段，则将创建 UploadFile 对象来访问底层文件，否则将对表单数据部分进行解析，并以原始字符串的形式提供。